# 03 — Demand Forecast Model (LightGBM)

**Project FORESIGHT — NorthBay Living**

This notebook:
1. Loads the cleaned weekly panel.
2. Runs a one-fold sanity check to confirm the model trains and predicts correctly.
3. Inspects feature importances.
4. Runs the full rolling-origin backtest (5 folds).
5. Summarises LightGBM WAPE vs seasonal-naive baseline WAPE.
6. Breaks WAPE down by product category.

> **The numbers produced in Section 5 must be recorded in `README.md`.**  
> Teammate 3 also needs them for the executive readout.

---
_Run all cells top-to-bottom. Do not skip sections._

## Section 1 — Setup & Data Loading

In [ ]:
import sys
import os

# Make sure the repo root is on the Python path
# (works whether you open the notebook from the notebooks/ folder or the repo root)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb

from src.config import PANEL_PATH, FORECAST_HORIZON_WEEKS
from src.forecast import (
    aggregate_to_weekly,
    add_weekly_features,
    get_weekly_feature_columns,
)
from src.metrics import wape, bias
from src.baseline import run_baseline_all_skus

# Load and prepare the weekly panel
panel = pd.read_parquet(PANEL_PATH)
weekly = aggregate_to_weekly(panel)
weekly = add_weekly_features(weekly)

print(f"Weekly panel shape: {weekly.shape}")
print(f"Date range : {weekly['week_start'].min().date()}  →  {weekly['week_start'].max().date()}")
print(f"Unique SKUs: {weekly['sku_id'].nunique()}")
print(f"Unique weeks: {weekly['week_start'].nunique()}")
print()
print("Column dtypes:")
print(weekly.dtypes)

## Section 2 — One-Fold Sanity Check

Quick verification that the model trains and produces sensible predictions  
**before** running the expensive 5-fold backtest.

In [ ]:
# Quick sanity check before running the full backtest
as_of = pd.Timestamp("2025-10-01")
train = weekly[weekly["week_start"] <= as_of]
test  = weekly[
    (weekly["week_start"] > as_of) &
    (weekly["week_start"] <= as_of + pd.Timedelta(weeks=8))
]

FEATURE_COLS = get_weekly_feature_columns()

# Prepare train
train_clean = train[FEATURE_COLS + ["units_sold"]].dropna()
X_train = train_clean[FEATURE_COLS]
y_train = train_clean["units_sold"]

# Prepare test
test_clean = test[FEATURE_COLS + ["units_sold", "sku_id", "week_start"]].dropna()
X_test = test_clean[FEATURE_COLS]
y_test = test_clean["units_sold"]

print(f"Train rows : {len(X_train):,}")
print(f"Test rows  : {len(X_test):,}")
print(f"Test SKUs  : {test_clean['sku_id'].nunique()}")
print()

# Train model
model = lgb.LGBMRegressor(
    objective="regression",
    num_leaves=63,
    learning_rate=0.05,
    n_estimators=500,
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train, categorical_feature=["category_code"])

# Predict and clip
preds = model.predict(X_test)
preds = np.clip(preds, 0, None)

fold_wape = wape(y_test.values, preds)
fold_bias = bias(y_test.values, preds)

print(f"Sanity check WAPE  : {fold_wape * 100:.2f}%")
print(f"Sanity check Bias  : {fold_bias:.2f} units/week")
print(f"Number of test rows: {len(y_test):,}")
print()
print("Sample predictions vs actuals (first 10 rows):")
sample = test_clean[["sku_id", "week_start"]].copy()
sample["actual"]    = y_test.values
sample["predicted"] = preds
print(sample.head(10).to_string(index=False))

## Section 3 — Feature Importance

Which features drive the LightGBM model most?  
Chart saved to `reports/model_feature_importance.png` for the executive readout.

In [ ]:
feat_imp = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(15).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Top 15 Feature Importances — LightGBM", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (split count)")
ax.invert_yaxis()
plt.tight_layout()

# Save chart — directory created if not present
import os
os.makedirs("../reports", exist_ok=True)
plt.savefig("../reports/model_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 10 features:")
print(feat_imp.head(10).to_string())

## Section 4 — Full Rolling-Origin Backtest

5-fold rolling-origin cross-validation.  
Each fold trains strictly on past data and tests on the next 8 weeks.  
**No future data touches training features — zero leakage.**

In [ ]:
from src.forecast import rolling_origin_backtest

print("Running 5-fold rolling-origin backtest...")
print("(Each fold will print its WAPE results below)")
print()

backtest_results = rolling_origin_backtest(weekly)

print()
print(f"Backtest complete. {len(backtest_results):,} rows across {backtest_results['fold'].nunique()} folds.")
print()
print("Sample backtest rows (first 20):")
print(backtest_results.head(20).to_string(index=False))

## Section 5 — WAPE Comparison Summary

> **⭐ Record the numbers printed by this cell in `README.md` and share them with Teammate 3.**

In [ ]:
# Compute overall WAPE for LightGBM vs baseline across all folds
overall_lgb_wape = wape(
    backtest_results["actual"].values,
    backtest_results["lgb_yhat"].values,
)
overall_baseline_wape = wape(
    backtest_results["actual"].values,
    backtest_results["baseline_yhat"].values,
)

improvement = (
    (overall_baseline_wape - overall_lgb_wape) / overall_baseline_wape * 100
    if overall_baseline_wape > 0
    else 0.0
)

print("=" * 50)
print("BACKTEST RESULTS SUMMARY")
print("=" * 50)
print(f"LightGBM WAPE      :  {overall_lgb_wape * 100:.2f}%")
print(f"Seasonal-Naive WAPE:  {overall_baseline_wape * 100:.2f}%")
print(f"Improvement        :  {improvement:.1f}%")
print()
if overall_lgb_wape < overall_baseline_wape:
    print("✅  LightGBM BEATS the seasonal-naive baseline.")
    print("    The model is trustworthy. Use it for the final forecast.")
else:
    print("⚠️   LightGBM does NOT beat the baseline.")
    print("    Ship the seasonal-naive baseline as the final forecast and report this honestly.")
print()
print("*** RECORD THESE NUMBERS IN README.md ***")
print(f"Baseline WAPE  :  {overall_baseline_wape * 100:.2f}%")
print(f"LightGBM WAPE  :  {overall_lgb_wape * 100:.2f}%")

# Per-fold breakdown
print()
print("Per-fold WAPE breakdown:")
fold_summary = (
    backtest_results
    .groupby("fold")
    .apply(
        lambda g: pd.Series({
            "cutoff_date":    g["cutoff_date"].iloc[0],
            "n_rows":         len(g),
            "lgb_wape_pct":   wape(g["actual"].values, g["lgb_yhat"].values) * 100,
            "base_wape_pct":  wape(g["actual"].values, g["baseline_yhat"].values) * 100,
        }),
        include_groups=False,
    )
    .reset_index()
)
fold_summary["winner"] = fold_summary.apply(
    lambda r: "✅ LGB" if r["lgb_wape_pct"] < r["base_wape_pct"] else "⚠️  Baseline", axis=1
)
print(fold_summary.to_string(index=False))

## Section 6 — WAPE by Category

Break down model performance by product category.  
If the model loses to baseline in a specific category, report that finding honestly.

In [ ]:
# Check if model beats baseline in each category separately
# Join category from the panel (sku_id → category mapping)
sku_category = (
    panel[["sku_id", "category"]]
    .drop_duplicates(subset="sku_id")
    .assign(category=lambda x: x["category"].astype(str))
)

backtest_with_meta = backtest_results.merge(
    sku_category,
    on="sku_id",
    how="left",
)

print(f"{'Category':<28}  {'LGB WAPE':>10}  {'Base WAPE':>10}  {'Winner':<14}")
print("-" * 70)

for cat in sorted(backtest_with_meta["category"].dropna().unique()):
    sub      = backtest_with_meta[backtest_with_meta["category"] == cat]
    cat_lgb  = wape(sub["actual"].values, sub["lgb_yhat"].values)
    cat_base = wape(sub["actual"].values, sub["baseline_yhat"].values)
    winner   = "✅ LGB" if cat_lgb < cat_base else "⚠️  Baseline"
    print(f"{cat:<28}  {cat_lgb*100:>9.1f}%  {cat_base*100:>9.1f}%  {winner}")

print()
print("Note: if the baseline wins for a category, document it in the EDA memo")
print("and executive readout. Honest reporting is required by the brief (Section 7.1).")

---
## Next Steps

1. **Record numbers from Section 5** in `README.md`.
2. **Message Teammate 3** with:
   - Baseline WAPE: `____%`
   - LightGBM WAPE: `____%`
3. Run the full pipeline to generate the parquet files Teammate 3 needs:
   ```bash
   python -m src.forecast   # → data/processed/forecasts.parquet
   python -m src.risk       # → data/processed/risk_scores.parquet
   ```
4. Confirm both files exist and commit everything to git.